# AeroPulse PM2.5 — 01B: CPCB Local-Drop Ingestion

Turn a manually exported CPCB CAAQMS file into the Phase-1 canonical PM2.5 contract, so the
multi-year history the roadmap wants can be added without reworking Phase 2.

## Objective

`01A` measured the gap: 19 months of data against a roadmap target of 3–5 years, with one
winter and one post-monsoon cycle. CPCB CAAQMS is the authoritative source for the missing
period, and Phase 1 left a `fetch_cpcb_hourly()` stub for it.

**This notebook is not the missing API.** It is the ingestion path for files a human has
already exported.

### Why there is no automated fetch here

`app.cpcbccr.com` is a captcha-gated JavaScript portal with no public historical REST
endpoint. Scraping it would breach the terms of use and would make the project's data
foundation depend on the stability of someone else's front-end. Neither is acceptable for
a layer everything downstream is built on, so the automated path is **deliberately not
implemented** rather than implemented badly.

The supported paths are, in order of preference:

1. A CPCB data-sharing agreement providing an agreed feed.
2. A manual portal export, dropped into the directory below.
3. A state pollution control board export in the same shape.

### How to use this

1. Export hourly PM2.5 from the CPCB portal for the stations and dates you want.
2. Drop the CSV or Excel files into `data/pm25/raw/cpcb_drop/`.
3. Run this notebook. It normalises, validates, and writes a canonical parquet.
4. Re-run Phase 1's persist step, then Phase 2. **No Phase-2 code changes are needed** —
   that is the point of normalising here.

With no files present the notebook prints the expected format and exits cleanly. It is
safe to run now, and it is meant to be run now, to prove the path works before anyone
depends on it.

In [1]:
# ============================================================================
# CONFIGURATION AND THE CANONICAL CONTRACT
# ============================================================================
from pathlib import Path
import os, json, re, hashlib
import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv = find_dotenv(usecwd=True)
if _dotenv:
    load_dotenv(_dotenv, override=True)

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
DROP_DIR = PROJECT_ROOT / "data" / "pm25" / "raw" / "cpcb_drop"
BASE_DIR = PROJECT_ROOT / "data" / "pm25" / "processed" / "base"
DROP_DIR.mkdir(parents=True, exist_ok=True)
BASE_DIR.mkdir(parents=True, exist_ok=True)

# EXACTLY the columns Phase 1's normalize_pm25() returns. This list is the whole
# contract: match it and Phase 2 needs no changes; deviate and the failure appears
# three notebooks away as an all-NaN feature.
PM25_COLUMNS = ["timestamp_utc", "location_id", "sensor_id", "station_name",
                "lat", "lon", "pm25", "pm25_unit", "source",
                "pm25_extreme_flag", "pm25_severe_flag"]

# CPCB exports are not consistent between the portal's report types, so column
# names are matched case-insensitively against these candidate lists rather than
# assumed.
COLUMN_ALIASES = {
    "timestamp": ["from date", "date", "datetime", "timestamp", "date and time",
                  "from_date", "to date"],
    "station_name": ["station", "station name", "location", "site", "site name"],
    "pm25": ["pm2.5", "pm25", "pm2_5", "pm 2.5", "pm2.5 (ug/m3)",
             "pm2.5 (µg/m³)", "pm2.5(ug/m3)"],
    "lat": ["latitude", "lat"],
    "lon": ["longitude", "lon", "long"],
    "city": ["city"],
    "state": ["state"],
}
# CPCB timestamps are recorded in IST with no timezone marker. Localising them
# as UTC would shift every reading by 5.5 hours and silently misalign the
# weather join, which is the single most likely way this ingestion goes wrong.
CPCB_TZ = "Asia/Kolkata"
EXTREME, SEVERE = 150.0, 90.0

print(f"Drop directory : {DROP_DIR}")
print(f"Canonical contract ({len(PM25_COLUMNS)} columns):")
for c in PM25_COLUMNS:
    print(f"  - {c}")

Drop directory : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/raw/cpcb_drop
Canonical contract (11 columns):
  - timestamp_utc
  - location_id
  - sensor_id
  - station_name
  - lat
  - lon
  - pm25
  - pm25_unit
  - source
  - pm25_extreme_flag
  - pm25_severe_flag


In [2]:
# ============================================================================
# STATION REGISTRY - coordinates the CPCB export usually does not carry
# ============================================================================
# Most CPCB report exports name the station but omit its latitude and longitude,
# and Phase 2 cannot join weather, fire or satellite data without coordinates.
# Two sources, in order:
#   1. the export itself, when it has lat/lon columns
#   2. the existing Phase-1 dataset, matched on station name
# Anything still unresolved is REJECTED rather than given a guessed coordinate -
# a station at the wrong place produces confident, wrong features.
# ============================================================================
registry = {}
BASE_FILE = BASE_DIR / "base_dataset.parquet"
if BASE_FILE.exists():
    known = (pd.read_parquet(BASE_FILE, columns=["station_name", "lat", "lon",
                                                 "location_id"])
             .drop_duplicates(subset=["station_name"]))
    for r in known.itertuples():
        registry[str(r.station_name).strip().lower()] = {
            "lat": float(r.lat), "lon": float(r.lon),
            "location_id": r.location_id}
    print(f"Loaded {len(registry)} known stations from the Phase-1 dataset.")
else:
    print("No Phase-1 dataset present; coordinates must come from the export itself.")

REGISTRY_FILE = DROP_DIR / "station_registry.csv"
if REGISTRY_FILE.exists():
    # Optional manual override: station_name,lat,lon
    extra = pd.read_csv(REGISTRY_FILE)
    for r in extra.itertuples():
        registry[str(r.station_name).strip().lower()] = {
            "lat": float(r.lat), "lon": float(r.lon),
            "location_id": f"cpcb::{str(r.station_name).strip()}"}
    print(f"Applied {len(extra)} overrides from {REGISTRY_FILE.name}.")
else:
    print(f"Optional: drop a {REGISTRY_FILE.name} with station_name,lat,lon to "
          "resolve stations the Phase-1 dataset does not know.")

Loaded 149 known stations from the Phase-1 dataset.
Optional: drop a station_registry.csv with station_name,lat,lon to resolve stations the Phase-1 dataset does not know.


In [3]:
# ============================================================================
# NORMALIZER - any CPCB export shape -> the canonical contract
# ============================================================================
def _match_column(columns, candidates):
    lowered = {re.sub(r"\s+", " ", str(c)).strip().lower(): c for c in columns}
    for cand in candidates:
        if cand in lowered:
            return lowered[cand]
    for key, original in lowered.items():          # substring fallback
        for cand in candidates:
            if cand in key:
                return original
    return None


def normalize_cpcb(raw, source_file):
    """Map one CPCB export into AeroPulse's canonical PM2.5 contract.

    Returns (frame, issues). The frame contains only rows that passed validation;
    `issues` records every row that did not and why, so a partial export is
    usable without the discards being silent.
    """
    issues = []
    cols = {k: _match_column(raw.columns, v) for k, v in COLUMN_ALIASES.items()}
    for required in ("timestamp", "pm25"):
        if cols[required] is None:
            raise ValueError(
                f"{source_file}: no column matched '{required}'. Columns present: "
                f"{list(raw.columns)[:15]}")

    out = pd.DataFrame()
    # IST -> UTC. Not tz_localize('UTC'), which would shift every reading 5.5h.
    ts = pd.to_datetime(raw[cols["timestamp"]], errors="coerce", dayfirst=True)
    out["timestamp_utc"] = ts.dt.tz_localize(CPCB_TZ, ambiguous="NaT",
                                             nonexistent="NaT").dt.tz_convert("UTC")
    out["pm25"] = pd.to_numeric(
        raw[cols["pm25"]].astype(str).str.replace(",", "", regex=False),
        errors="coerce")

    if cols["station_name"]:
        out["station_name"] = raw[cols["station_name"]].astype(str).str.strip()
    else:
        # Portal exports named per station carry it in the filename, not a column.
        out["station_name"] = source_file.stem
        issues.append({"file": source_file.name, "issue": "no station column",
                       "detail": f"used filename '{source_file.stem}'", "rows": len(raw)})

    if cols["lat"] and cols["lon"]:
        out["lat"] = pd.to_numeric(raw[cols["lat"]], errors="coerce")
        out["lon"] = pd.to_numeric(raw[cols["lon"]], errors="coerce")
    else:
        out["lat"] = np.nan
        out["lon"] = np.nan

    resolved = out["station_name"].str.strip().str.lower().map(registry)
    out["lat"] = out["lat"].fillna(resolved.map(
        lambda d: d["lat"] if isinstance(d, dict) else np.nan))
    out["lon"] = out["lon"].fillna(resolved.map(
        lambda d: d["lon"] if isinstance(d, dict) else np.nan))
    out["location_id"] = resolved.map(
        lambda d: d["location_id"] if isinstance(d, dict) else np.nan)
    out["location_id"] = out["location_id"].fillna(
        "cpcb::" + out["station_name"].str.strip())
    out["sensor_id"] = out["location_id"].astype(str) + "::pm25"
    out["pm25_unit"] = "ug/m3"
    out["source"] = "CPCB"          # per-row provenance stays traceable

    n0 = len(out)
    bad_ts = out.timestamp_utc.isna()
    bad_pm = out.pm25.isna()
    # India's record hourly PM2.5 readings reach the high hundreds; 2000 is a
    # generous ceiling that still catches a unit or a decimal-place error.
    out_of_range = (out.pm25 < 0) | (out.pm25 > 2000)
    no_coords = out.lat.isna() | out.lon.isna()
    # The Phase-1 India bbox. A station outside it is a coordinate error.
    outside_india = ~(out.lat.between(6.0, 38.0) & out.lon.between(66.0, 98.0))
    for name, mask in [("unparseable timestamp", bad_ts), ("missing PM2.5", bad_pm),
                       ("PM2.5 out of range", out_of_range),
                       ("unresolved coordinates", no_coords),
                       ("coordinates outside India", outside_india & ~no_coords)]:
        if mask.sum():
            issues.append({"file": source_file.name, "issue": name,
                           "detail": "rejected", "rows": int(mask.sum())})

    keep = ~(bad_ts | bad_pm | out_of_range | no_coords | outside_india)
    out = out.loc[keep].copy()
    out["pm25_extreme_flag"] = out.pm25 >= EXTREME
    out["pm25_severe_flag"] = out.pm25 >= SEVERE
    # Hourly grid, matching Phase 1. Duplicate station-hours keep the first.
    out["timestamp_utc"] = out["timestamp_utc"].dt.floor("h")
    before = len(out)
    out = out.drop_duplicates(subset=["location_id", "timestamp_utc"], keep="first")
    if before != len(out):
        issues.append({"file": source_file.name, "issue": "duplicate station-hours",
                       "detail": "kept first", "rows": before - len(out)})
    print(f"  {source_file.name:44} {n0:>8,} raw -> {len(out):>8,} accepted")
    return out[PM25_COLUMNS], issues


print("Normalizer ready.")

Normalizer ready.


In [4]:
# ============================================================================
# INGEST whatever is in the drop directory
# ============================================================================
files = sorted([p for p in DROP_DIR.iterdir()
                if p.suffix.lower() in (".csv", ".xlsx", ".xls")
                and p.name != REGISTRY_FILE.name])

EXPECTED_FORMAT = """
Expected shape of a dropped file (column names are matched case-insensitively
and several CPCB variants are recognised):

    From Date,Station,PM2.5,Latitude,Longitude
    01-11-2023 00:00,Anand Vihar, Delhi - DPCC,412.5,28.6469,77.3159
    01-11-2023 01:00,Anand Vihar, Delhi - DPCC,398.2,28.6469,77.3159

Rules:
  - timestamps are read as IST and converted to UTC
  - Latitude/Longitude may be omitted IF the station name matches one already in
    the Phase-1 dataset, or appears in station_registry.csv
  - one row per station-hour; duplicates keep the first
  - PM2.5 in ug/m3
"""

if not files:
    print(f"No files in {DROP_DIR}\n")
    print(EXPECTED_FORMAT)
    print("This is the expected state until a CPCB export is available. The "
          "notebook is complete and exits cleanly; nothing downstream changes.")
    combined = pd.DataFrame(columns=PM25_COLUMNS)
    all_issues = []
else:
    print(f"Found {len(files)} file(s):\n")
    frames, all_issues = [], []
    for f in files:
        raw = (pd.read_csv(f, low_memory=False) if f.suffix.lower() == ".csv"
               else pd.read_excel(f))
        frame, issues = normalize_cpcb(raw, f)
        frames.append(frame)
        all_issues.extend(issues)
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
        columns=PM25_COLUMNS)
    combined = combined.drop_duplicates(subset=["location_id", "timestamp_utc"],
                                        keep="first")
    print(f"\nCombined: {len(combined):,} rows, "
          f"{combined.location_id.nunique()} stations")
    if all_issues:
        print("\nRejections and adjustments:")
        display(pd.DataFrame(all_issues))

No files in /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/raw/cpcb_drop


Expected shape of a dropped file (column names are matched case-insensitively
and several CPCB variants are recognised):

    From Date,Station,PM2.5,Latitude,Longitude
    01-11-2023 00:00,Anand Vihar, Delhi - DPCC,412.5,28.6469,77.3159
    01-11-2023 01:00,Anand Vihar, Delhi - DPCC,398.2,28.6469,77.3159

Rules:
  - timestamps are read as IST and converted to UTC
  - Latitude/Longitude may be omitted IF the station name matches one already in
    the Phase-1 dataset, or appears in station_registry.csv
  - one row per station-hour; duplicates keep the first
  - PM2.5 in ug/m3

This is the expected state until a CPCB export is available. The notebook is complete and exits cleanly; nothing downstream changes.


In [5]:
# ============================================================================
# VALIDATION GATES - fail here, not in Phase 2
# ============================================================================
gates, failures = {}, []


def gate(name, ok, detail=""):
    gates[name] = {"pass": bool(ok), "detail": detail}
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:38} {detail}")
    if not ok:
        failures.append(name)


if combined.empty:
    print("No rows to validate (empty drop directory).")
else:
    print("Ingestion gates:")
    gate("canonical columns present",
         list(combined.columns) == PM25_COLUMNS,
         f"{len(combined.columns)} columns")
    gate("timestamps are tz-aware UTC",
         str(combined.timestamp_utc.dt.tz) == "UTC")
    gate("no duplicate station-hours",
         not combined.duplicated(subset=["location_id", "timestamp_utc"]).any())
    gate("PM2.5 within plausible range",
         combined.pm25.between(0, 2000).all(),
         f"{combined.pm25.min():.1f} .. {combined.pm25.max():.1f} ug/m3")
    gate("all coordinates resolved",
         combined[["lat", "lon"]].notna().all().all())
    gate("every row attributed to CPCB",
         (combined.source == "CPCB").all())
    # The whole reason for this notebook: does the drop actually extend history?
    if BASE_FILE.exists():
        existing_min = pd.read_parquet(BASE_FILE,
                                       columns=["timestamp_utc"]).timestamp_utc.min()
        existing_min = pd.Timestamp(existing_min).tz_convert("UTC")
        extends = combined.timestamp_utc.min() < existing_min
        gate("extends the existing history", extends,
             f"CPCB starts {combined.timestamp_utc.min().date()}, existing starts "
             f"{existing_min.date()}")
    if failures:
        raise RuntimeError(f"CPCB ingestion gates failed: {failures}")
    print("\nAll gates passed.")

No rows to validate (empty drop directory).


In [6]:
# ============================================================================
# PERSIST - a separate canonical file, not an in-place edit of Phase 1
# ============================================================================
# Written alongside the Phase-1 output rather than merged into it. Phase 1 stays
# reproducible from its own sources, and the CPCB contribution stays separately
# auditable - which matters because it is manually sourced.
# ============================================================================
CPCB_FILE = BASE_DIR / "pm25_cpcb_canonical.parquet"
report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "drop_directory": str(DROP_DIR.relative_to(PROJECT_ROOT)),
    "files_ingested": [f.name for f in files],
    "rows": int(len(combined)),
    "stations": int(combined.location_id.nunique()) if not combined.empty else 0,
    "window": ([str(combined.timestamp_utc.min()), str(combined.timestamp_utc.max())]
               if not combined.empty else None),
    "issues": all_issues,
    "gates": gates,
    "canonical_columns": PM25_COLUMNS,
    "timezone_handling": "CPCB timestamps read as Asia/Kolkata and converted to "
                         "UTC; localising them as UTC would shift every reading "
                         "by 5.5 hours",
    "automation_status": "NOT AUTOMATED BY DESIGN - app.cpcbccr.com is a "
                         "captcha-gated JS portal with no public historical API. "
                         "Scraping it would breach the terms of use and make the "
                         "data foundation depend on a third-party front-end.",
    "next_steps": [
        "re-run 01_pm25_phase1 so the CPCB rows join weather, fire and satellite",
        "re-run 02_pm25_phase2 to rebuild features over the extended window",
        "re-run 01A to confirm the seasonal-cycle count actually improved",
    ],
}

if combined.empty:
    print(f"Nothing written - {DROP_DIR.name} is empty.")
    print("The scaffold is in place: drop a CPCB export here and re-run.")
else:
    combined.to_parquet(CPCB_FILE, index=False)
    h = hashlib.sha256(CPCB_FILE.read_bytes()).hexdigest()[:16]
    report["output_file"] = str(CPCB_FILE.relative_to(PROJECT_ROOT))
    report["sha256_16"] = h
    print(f"Wrote {CPCB_FILE} ({CPCB_FILE.stat().st_size/1e6:.1f} MB, sha {h})")
    print(f"  {len(combined):,} rows, {combined.location_id.nunique()} stations")
    print(f"  {combined.timestamp_utc.min()} -> {combined.timestamp_utc.max()}")

with open(BASE_DIR / "cpcb_ingestion_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"\nWrote report: {BASE_DIR / 'cpcb_ingestion_report.json'}")
print("\nTo extend the dataset:")
for i, s in enumerate(report["next_steps"], 1):
    print(f"  {i}. {s}")

Nothing written - cpcb_drop is empty.
The scaffold is in place: drop a CPCB export here and re-run.

Wrote report: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/base/cpcb_ingestion_report.json

To extend the dataset:
  1. re-run 01_pm25_phase1 so the CPCB rows join weather, fire and satellite
  2. re-run 02_pm25_phase2 to rebuild features over the extended window
  3. re-run 01A to confirm the seasonal-cycle count actually improved
